# SHA Flight-Service Patterns Around China's 2026 National Day Holiday

**Approved proposal questions:**
1. How did flight-service volume and cancellation proportions vary before, during, and after the holiday?
2. Were cancellations and extreme recorded departure-time gaps concentrated on particular dates or departure hours?
3. What does the October 10 adjusted workday reveal about weekday/weekend labels?

This notebook is documented and reproducible. It uses the supplied `sha_master.csv`. A ≥60-minute scheduled-to-actual departure gap is a pre-defined recorded timestamp feature, not a verified operational delay.

**Scope and data boundary:** The unit of observation is a flight-event record. This is not a passenger-demand, congestion, emissions, or causal study.

In [ ]:
from pathlib import Path
import csv
from collections import defaultdict
from datetime import date, timedelta, datetime

DATA = Path('sha_master.csv')
if not DATA.exists():
    DATA = Path('..') / 'sha_master.csv'
with DATA.open(newline='', encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))
for r in rows:
    r['_date'] = date.fromisoformat(r['local_date'])
    r['_hour'] = int(r['local_hour'])
print(f'{len(rows):,} records; {len({r["local_date"] for r in rows})} dates')
print(f'Unique event IDs: {len({r["event_id"] for r in rows})}')

## Methods and definitions

- **Known status:** status is neither blank nor `unknown`.
- **Cancellation proportion:** cancellations ÷ known-status records.
- **Period comparison:** three 7-day windows with all 7 dates present.
- **Extreme recorded gap:** `actual_departure_utc - scheduled_departure_utc >= 60` minutes among known-status, non-cancelled departures with both timestamps.
- **Hour bands:** morning 06–11, afternoon 12–16, evening 17–22, and night/other 23–05.
- The source records `timing_semantics_verified=false` throughout, so timestamp differences are not called operational delays.

In [ ]:
def known(r):
    return r['status_display'].strip().lower() not in ('', 'unknown')
def cancelled(r):
    return r['status_display'].strip().lower() == 'canceled'
def pct(n,d):
    return 100*n/d if d else 0

def summarize(rs):
    k = [r for r in rs if known(r)]
    c = sum(cancelled(r) for r in k)
    return {'events':len(rs), 'known':len(k), 'unknown':len(rs)-len(k), 'cancelled':c,
            'cancellation_pct':pct(c,len(k))}

holiday = date(2026,10,1)
periods = [('Pre-holiday', holiday-timedelta(days=7), holiday-timedelta(days=1)),
           ('National Day', holiday, holiday+timedelta(days=6)),
           ('Post-holiday', holiday+timedelta(days=7), holiday+timedelta(days=13))]
for label,start,end in periods:
    subset=[r for r in rows if start <= r['_date'] <= end]
    s=summarize(subset)
    print(label, start, end, s, f"events/day={len(subset)/7:.1f}")

## RQ1 — Calendar phase and daily trend

The period table is intentionally supplemented with daily values. Period averages can conceal a high-cancellation date within a generally low-cancellation window. The following chart plots both flight-event volume and cancellation proportion for every available date.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.dates import DayLocator, DateFormatter
from collections import defaultdict
by_date=defaultdict(list)
for r in rows: by_date[r['_date']].append(r)
daily=[(d,summarize(rs)) for d,rs in sorted(by_date.items())]
fig,ax=plt.subplots(2,1,figsize=(9,6.5),sharex=True)
dates=[d for d,_ in daily]
ax[0].plot(dates,[s['events'] for d,s in daily],marker='o',ms=3)
ax[0].set_ylabel('Flight events'); ax[0].set_title('Daily flight-event volume')
ax[1].plot(dates,[s['cancellation_pct'] for d,s in daily],marker='o',ms=3,color='#e07a32')
ax[1].axvspan(holiday, holiday+timedelta(days=7), color='#e07a32', alpha=.12)
ax[1].set_ylabel('Cancellation (%)'); ax[1].set_xlabel('Local date')
ax[1].xaxis.set_major_locator(DayLocator(interval=3)); ax[1].xaxis.set_major_formatter(DateFormatter('%b %-d'))
fig.tight_layout(); plt.show()
print('Calendar qualifiers: 25–27 Sep = Mid-Autumn public holiday; 1–7 Oct = National Day holiday; 10 Oct = adjusted workday.')

## RQ2 — Concentration on dates and hours

The numerator is a date/hour count of **recorded** extreme gaps. It is compared with the eligible known-status, non-cancelled departure denominator. Cancellations are analysed separately and are not eligible for departure-gap calculation.

In [ ]:
from datetime import datetime

def parse_dt(value):
    return datetime.fromisoformat(value.replace('Z','+00:00'))
eligible=[]
missing=0
for r in rows:
    if r['direction']!='departure' or not known(r) or cancelled(r):
        continue
    if not r['scheduled_departure_utc'] or not r['actual_departure_utc']:
        missing += 1; continue
    gap=(parse_dt(r['actual_departure_utc'])-parse_dt(r['scheduled_departure_utc'])).total_seconds()/60
    eligible.append((r,gap))

date_ext=[]
for d in sorted({r['_date'] for r,g in eligible}):
    gs=[g for r,g in eligible if r['_date']==d]
    date_ext.append((d,len(gs),sum(g>=60 for g in gs),pct(sum(g>=60 for g in gs),len(gs))))
print('eligible',len(eligible),'missing',missing,'extreme',sum(g>=60 for r,g in eligible))
print('Top date rows:',sorted(date_ext,key=lambda x:(-x[3],-x[2]))[:5])

bands=defaultdict(list)
for r,g in eligible:
    h=r['_hour']; band='Morning (06–11)' if 6<=h<=11 else ('Afternoon (12–16)' if 12<=h<=16 else ('Evening (17–22)' if 17<=h<=22 else 'Night/other'))
    bands[band].append(g)
for band,gs in sorted(bands.items()):
    print(band, 'n=',len(gs),'extreme=',sum(g>=60 for g in gs),'pct=',round(pct(sum(g>=60 for g in gs),len(gs)),2))

The date-level view is not a test of recurring causation. October 10 is examined separately because ordinary weekday/weekend labels are not sufficient descriptions in China's adjusted-workday calendar.

In [ ]:
def cancellation_table(rs):
    s=summarize(rs)
    print(s)
    return s

print('Top cancellation dates')
top=sorted(daily,key=lambda ds:(-ds[1]['cancellation_pct'],ds[0].isoformat()))[:5]
for d,s in top:
    print(d.isoformat(), s)

print('October 10 adjusted workday')
cancellation_table([r for r in rows if r['_date']==date(2026,10,10)])
print('Ordinary weekends')
for d,s in daily:
    r=by_date[d][0]
    if r['calendar_day_type']=='weekend':
        print(d.isoformat(),s)

## RQ3 — Adjusted workday case study

October 10 was officially a Saturday adjusted workday. There is only one adjusted workday in the dataset, and exactly three ordinary weekend dates. Therefore, all apparent contrasts are observations about these particular dates—not estimates of a general adjusted-workday effect.

In [ ]:
adjusted = summarize([r for r in rows if r['_date']==date(2026,10,10)])
ordinary = [(d,s) for d,s in daily if by_date[d][0]['calendar_day_type']=='weekend']
labels=['10 Oct\nadjusted']+[d.strftime('%d %b')+'\nweekend' for d,s in ordinary]
values=[adjusted['cancellation_pct']]+[s['cancellation_pct'] for d,s in ordinary]
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(labels,values,color=['#be4b48','#2878b5','#2878b5','#2878b5'])
ax.set_ylabel('Cancellation (%)'); ax.set_title('October 10 compared with ordinary weekend dates')
ax.grid(axis='y',alpha=.25); fig.tight_layout(); plt.show()
print('Events:', adjusted['events'], '| known:', adjusted['known'], '| cancelled:', adjusted['cancelled'], '| rate:', adjusted['cancellation_pct'])

## Conclusions

The notebook reproduces the three approved questions from the source CSV. Period comparisons show that calendar status is an important qualifier. Cancellation and extreme-gap indicators are concentrated in specific dates and tend to differ across hour bands. The October 10 result is consistent with the conclusion that conventional weekday/weekend labels omit make-up workdays, but it cannot identify a causal or general effect. Remaining analysis should combine this exploratory evidence with passenger demand, operational records, weather, and more years of airport data.